# Train `ASCIIVAE` on Google Colab

Reads the ASCII shards produced by `notebooks/collect_gvgai_ascii_colab.ipynb` and trains `world_model.model.net.ascii_vae.ASCIIVAE` via `world_model/train_ascii_vae.py`. Both notebooks share the same Drive layout:

```
MyDrive/DecoupliWo/
  data/transitions/{train,test}/<game>/<variant>/shard_XXXXX/  # pushed per-shard by collection
  checkpoints/ascii_vae/vae.pt                                  # written here
  runs/ascii_vae/<timestamp>/                                   # TensorBoard here
```

## Runtime

1. `Runtime` → `Change runtime type` → **GPU** → pick **A100** or **H100** (Pro / Pro+ only). T4 also works; the model is tiny.
2. Enable **High-RAM** if offered.

## Simultaneous collection + training (two tabs)

This notebook is designed to run **in parallel** with `collect_gvgai_ascii_colab.ipynb` on a separate Colab tab (CPU runtime for collection, GPU runtime here). The collection notebook round-robins across games and pushes each finalized `shard_XXXXX/` to Drive as soon as it's written, so each training mini-batch naturally mixes games once shards from all games have landed on Drive.

You can start this notebook any time after the collection tab has produced at least one `shard_*/` under `MyDrive/DecoupliWo/data/transitions/train/`. The trainer rescans Drive every `--refresh_every` steps (see the training cell near the bottom) and hot-grows its dataset with any new shards that have appeared — no need to restart training.

Running two runtimes concurrently on the same account requires Colab Pro or Pro+; the free tier will evict the older session.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Canonical paths (shared with the collection notebook)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_DATA_DIR = DRIVE_ROOT / 'data' / 'transitions'
DRIVE_CKPT_DIR = DRIVE_ROOT / 'checkpoints' / 'ascii_vae'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs' / 'ascii_vae'
DRIVE_GVGAI_TAR = DRIVE_ROOT / 'gvgai.tar.gz'

LOCAL_DATA_DIR = REPO_DIR / 'data' / 'transitions'
LOCAL_RUNS_DIR = Path('/content/runs/ascii_vae')

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA_DIR.is_dir() and any(DRIVE_DATA_DIR.rglob('shard_*')), (
    f'No shards under {DRIVE_DATA_DIR}. Run collect_gvgai_ascii_colab.ipynb first.'
)
assert DRIVE_GVGAI_TAR.is_file(), (
    f'Expected {DRIVE_GVGAI_TAR} on Drive (uploaded once during the collection notebook setup). '
    f'See notebooks/collect_gvgai_ascii_colab.ipynb for the one-time tar command.'
)
print('OK drive data dir :', DRIVE_DATA_DIR)
print('OK ckpt dir      :', DRIVE_CKPT_DIR)
print('OK drive runs dir:', DRIVE_RUNS_DIR)
print('OK local runs dir:', LOCAL_RUNS_DIR)
print('OK gvgai bundle  :', DRIVE_GVGAI_TAR, f'({DRIVE_GVGAI_TAR.stat().st_size / 1e6:.1f} MB)')

## 4. Clone the repo

In [ ]:
import subprocess

if REPO_DIR.is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 5. Install minimal dependencies

The full `requirements.txt` drags in Atari / OCAtari / stable-baselines3 / diffusers / etc. None of those are needed to train `ASCIIVAE`. We only need `tensorboard`, `tqdm`, and PyTorch + numpy (pre-installed on Colab GPU runtimes).

Do **not** pin numpy. The current Colab image ships numpy 2.x and every C extension on it (jax, cupy, opencv, torch plugins) is built against that ABI — downgrading to 1.x produces the classic `numpy.dtype size changed, may indicate binary incompatibility` import error.

In [ ]:
!pip install --quiet tensorboard tqdm

## 6. Prepare the GVGAI renderer (required for `--render_rgb`)

To log RGB *original | reconstruction* panels to TensorBoard during training, the trainer needs a running JVM renderer (`tracks.singlePlayer.rendering.AsciiRenderServer`). The class isn't in `gvgai.tar.gz` on Drive, but the Java source lives in this repo at `gvgai_java_stubs/src/tracks/singlePlayer/rendering/AsciiRenderServer.java` and only depends on `core.*` classes that **are** in the tar. So the next cell:

1. Extracts `gvgai.tar.gz` from Drive into `/content/DecoupliWo/gvgai/` (idempotent; skips if already extracted).
2. Copies the renderer stub from the cloned repo into `gvgai/src/tracks/singlePlayer/rendering/` so the GVGAI tree sees it.
3. Installs `default-jdk` if `javac` isn't on the VM.
4. Compiles the stub against the extracted `core.*` classes, producing `gvgai/out/production/gvgai/tracks/singlePlayer/rendering/AsciiRenderServer.class`.

**Skip this section if you don't want RGB panels.** The trainer will still run; just omit `--render_rgb` from Cell 20 and no JVM renderer is spawned. If you keep `--render_rgb` but skip this cell, the trainer prints a warning per game and proceeds without RGB logging (scalars still work).

In [ ]:
import shutil
import subprocess
import tarfile

GVGAI_DIR = REPO_DIR / 'gvgai'
GVGAI_CLASSPATH = GVGAI_DIR / 'out' / 'production' / 'gvgai'
RENDERER_STUB = REPO_DIR / 'gvgai_java_stubs' / 'src' / 'tracks' / 'singlePlayer' / 'rendering' / 'AsciiRenderServer.java'
RENDERER_SRC = GVGAI_DIR / 'src' / 'tracks' / 'singlePlayer' / 'rendering' / 'AsciiRenderServer.java'
RENDERER_CLASS = GVGAI_CLASSPATH / 'tracks' / 'singlePlayer' / 'rendering' / 'AsciiRenderServer.class'

if not (GVGAI_CLASSPATH / 'core').is_dir():
    with tarfile.open(DRIVE_GVGAI_TAR, 'r:gz') as tf:
        tf.extractall(REPO_DIR)
    print(f'extracted gvgai -> {GVGAI_DIR}')
else:
    print(f'gvgai already extracted -> {GVGAI_DIR}')

assert RENDERER_STUB.is_file(), (
    f'Missing renderer stub at {RENDERER_STUB}. This file ships with the repo; '
    f'make sure Cell 8 cloned the correct branch.'
)
RENDERER_SRC.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(RENDERER_STUB, RENDERER_SRC)
print(f'staged {RENDERER_STUB.name} -> {RENDERER_SRC}')

if shutil.which('javac') is None:
    print('javac not found; installing default-jdk (one-time per Colab session)')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'default-jdk'], check=True)

subprocess.run(
    ['javac', '-cp', str(GVGAI_CLASSPATH), '-d', str(GVGAI_CLASSPATH), str(RENDERER_SRC)],
    check=True,
)
assert RENDERER_CLASS.is_file(), f'javac ran but {RENDERER_CLASS} was not produced'
print(f'OK renderer class: {RENDERER_CLASS}  ({RENDERER_CLASS.stat().st_size} bytes)')

## 7. Sync shards Drive → local SSD

Reading `.npy` files via `mmap_mode='r'` over the Drive FUSE mount is extremely slow (per-page network round-trips). One-time `rsync` to the Colab VM's local NVMe takes a few minutes and then training reads are fast. `rsync` is incremental: subsequent runs only copy new/changed shards.

In [ ]:
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 "{DRIVE_DATA_DIR}/" "{LOCAL_DATA_DIR}/"
print('train shards:', len(list((LOCAL_DATA_DIR / 'train').rglob('shard_*'))))
print('test  shards:', len(list((LOCAL_DATA_DIR / 'test').rglob('shard_*'))))

## 8. Sanity check: one forward pass

In [ ]:
import sys
import torch

sys.path.insert(0, str(REPO_DIR))

from world_model.ascii.constants import CANVAS_H, CANVAS_W, VOCAB_SIZE
from world_model.model.net.ascii_vae import ASCIIVAE
from world_model.train_ascii_vae import AllAsciiFramesDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ds = AllAsciiFramesDataset(LOCAL_DATA_DIR / 'train', CANVAS_H, CANVAS_W)
print(f'train frames = {len(ds):,}   canvas = {CANVAS_H}x{CANVAS_W}   vocab = {VOCAB_SIZE}')

vae = ASCIIVAE().to(device)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, mu, logvar = vae(ids)
print('ids    ', tuple(ids.shape), ids.dtype)
print('logits ', tuple(logits.shape))
print('mu     ', tuple(mu.shape))
print('params ', sum(p.numel() for p in vae.parameters()))

## 9. Launch TensorBoard

TensorBoard reads event files from **local SSD** (`LOCAL_RUNS_DIR`), not Drive. Writing event files through the Drive FUSE mount is unreliable — small append-style writes from `SummaryWriter` are batched, synced on a delay, and TB's reloader silently drops partial reads. Symptom: TB shows old runs but the *current* run appears empty or stale for minutes at a time.

The training script still persists events to Drive by rsyncing the run dir into `DRIVE_RUNS_DIR` on every `--save_every` tick (and once more at end-of-run), via the `--log_mirror` flag. So Drive keeps a durable copy for cross-session viewing, and TB gets fresh data within a few seconds.

After training starts, scroll to the bottom of the **Runs** list in the TB UI to find the new timestamped run, and hit the TB **refresh** button (circular arrow, top-right of the TB panel) if you want an immediate update before the next poll.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{LOCAL_RUNS_DIR}" --reload_multifile true --reload_interval 5

## 10. Train

Flag notes:

- `--output_dir` points at Drive so `vae.pt` survives VM termination.
- `--log_dir` points at **local SSD** (`LOCAL_RUNS_DIR`) so `SummaryWriter` writes are fast and TB reads are consistent. `--log_mirror` rsyncs the timestamped run dir into `DRIVE_RUNS_DIR` on every `--save_every` tick (and at end-of-run) so you keep a durable copy across sessions.
- Tune `--batch_size` / `--num_workers` for your GPU. H100/A100: 256 / 4 is comfortable. T4: drop to 128 / 2.
- **Match `--epochs` × steps-per-epoch to `--max_train_steps`** so the shorter of the two doesn't truncate training. With few thousand frames you want `--epochs` large (e.g. 300) and `--max_train_steps` as the real cap.
- Default `--max_train_steps 1000000` is sized for a full ~3 h H100 session so that the trainer keeps looping over the live-growing dataset (via `--refresh_every`) until either the session times out or you hit stop. The latest `--save_every`-step checkpoint on Drive is always a usable `vae.pt`.
- `--render_rgb` logs `val/rgb_reconstruction` images every `--save_every` steps: one fixed val frame per game, encoded+decoded+argmaxed, then rendered to RGB via a JVM `AsciiRenderServer` spawned per game. `--render_games` picks which games appear; `--gvgai_root` points at the extracted gvgai tree from Section 6. Remove these three flags to disable RGB logging (scalars still train normally).

### Online streaming from Drive

Two flags keep the trainer in lock-step with the collection tab:

- `--sync_from <drive path>`: before each refresh, `rsync -a` this path into `--train_dir`. Point it at `DRIVE_DATA_DIR/train` so we inherit shards pushed by the uploader over in the collection tab.
- `--refresh_every <N>`: every N optimizer steps, `_maybe_sync` then rescan `--train_dir` for new `shard_*` directories. `AllAsciiFramesDataset.refresh()` appends newly-seen shards; the `DataLoader` is rebuilt so workers pick up the new shards and batches begin mixing the newly-arrived games. `0` disables (sequential mode).

Turnaround time for a new frame from collector → gradient update is roughly:

```
java-side shard flush  +  uploader poll interval  +  refresh_every * step time
```

With `CHUNK_SIZE=5000`, `UPLOADER_POLL_SECONDS=10`, `refresh_every=500`, and ~20 ms/step on an A100, that's a few minutes — well below the collection budget for any real run.

### Workflow

1. Open a **second Colab tab** with `collect_gvgai_ascii_colab.ipynb` on a **CPU** runtime and start it collecting. That notebook writes shards to its VM's local SSD and uploads each one to Drive as it's finalized.
2. As soon as any shards appear under `MyDrive/DecoupliWo/data/transitions/train/` (usually within a minute of the first chunk finishing), start this notebook on a **GPU** runtime. Cell 6 asserts shards are present; Cell 12 does the initial Drive → local-SSD sync.
3. Leave both tabs running. Training stops at `--max_train_steps`; collection stops at its own frame budget. If one finishes first, the other keeps going without intervention.

In [ ]:
!python -m world_model.train_ascii_vae \
  --train_dir "{LOCAL_DATA_DIR}/train" \
  --val_dir   "{LOCAL_DATA_DIR}/test"  \
  --output_dir "{DRIVE_CKPT_DIR}" \
  --log_dir    "{LOCAL_RUNS_DIR}" \
  --log_mirror "{DRIVE_RUNS_DIR}" \
  --sync_from  "{DRIVE_DATA_DIR}/train" \
  --refresh_every 500 \
  --batch_size 256 \
  --num_workers 4 \
  --epochs 1000000 \
  --max_train_steps 1000000 \
  --save_every 5000 \
  --validation_every 1000 \
  --log_every 100 \
  --render_rgb \
  --render_games aliens,chopper,waves \
  --gvgai_root "{GVGAI_DIR}"

## 11. Verify saved checkpoint

In [ ]:
from world_model.model.net.ascii_vae import load_ascii_vae

ckpt = DRIVE_CKPT_DIR / 'vae.pt'
assert ckpt.is_file(), f'no checkpoint at {ckpt}'

vae = load_ascii_vae(ckpt).to(device)
vae.train(False)
ids = torch.stack([ds[i] for i in range(4)]).to(device)
logits, _, _ = vae(ids)
pred = ASCIIVAE.logits_to_ids(logits)
acc = (pred == ids).float().mean().item()
print(f'checkpoint    : {ckpt}  ({ckpt.stat().st_size / 1e6:.2f} MB)')
print(f'scaling_factor: {vae.scaling_factor:.4f}')
print(f'per-cell acc  : {acc:.4f} on 4 sample frames')